# 03 · Evaluation: what improved, and what still fails?
**Question:** What evidence supports the project, and what does it not establish?

We report three different things separately: role classification, summary overlap, and source/decision review. The tables below are **recorded results from previous runs**, not scores produced by this notebook. No training or test-set tuning happens here.

## Setup — run once
The project uses an isolated Python environment so its Hugging Face packages do not conflict with Colab's installed Gradio/Diffusers.
`%%project` means “run this ordinary Python cell in that environment.” Variables persist between these cells.
The first setup downloads dependencies and can take several minutes. No token is needed for the demo or downloaded trained model.
For the gated role dataset, accept its access conditions and add `HF_TOKEN` in Colab Secrets; never paste it into a cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/legal-judgment-summarizer' if Path('/content').exists() else Path.cwd()))
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Mananfrfr/legal-judgment-summarizer.git', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
from notebook_runtime import setup
setup(ROOT)


In [ ]:
%%project
from pathlib import Path
import os, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path(os.environ['PROJECT_ROOT'])


## 1. Role classification
Accuracy counts all correct labels. **Macro F1** gives every role equal weight, including rare ones. Weighted F1 gives common roles more influence. These numbers are on the development split; they are not unseen-document legal accuracy.

In [ ]:
%%project
roles = pd.read_csv(ROOT / 'evaluation/role_results.csv')
display(roles.round(4))
ax = roles.plot.barh(x='model', y='macro_f1', figsize=(9, 5), legend=False, title='Recorded DEV macro F1')
ax.set_xlabel('Macro F1 (0–1)')


## 2. Summarization on the original frozen protocol
ROUGE measures overlap with reference wording; it does not verify that the selected court decision is current or correct. The recorded Stage 3.6 test used 100 official cases, 120 sampled candidates and a 400-word budget. It had already been run once before this consolidation.
No model wins every metric. Report the entire comparison, not just the best number. Role features did not consistently improve the results.

In [ ]:
%%project
results = pd.read_csv(ROOT / 'evaluation/evaluation_results.csv')
display(results.round(4))
results.plot.bar(x='method', y=['rouge1_f1', 'rouge2_f1', 'rougeL_f1'], figsize=(10, 4),
                 title='Recorded Stage 3.6 official test (100 cases)')


## 3. Why those ROUGE scores cannot be assigned to today's app
The app now scores the full extracted judgment in overlapping windows and builds structured sections with evidence checks. That changes the inference protocol. The Stage 3.6 numbers describe the earlier fixed-budget experiment, not the current app.
All extracted sentences being scored means coverage of extracted candidates; it does not prove flawless PDF extraction or complete summaries.

In [ ]:
%%project
review = json.loads((ROOT / 'evaluation/current_pipeline_review.json').read_text())
display(pd.DataFrame([{'file': x['file'], 'scored': x['scored'], 'extracted': x['extracted_sentences'],
    'exact_evidence': x['exact_evidence'], 'critical_decision_failure': x['critical_decision_failure']}
    for x in review['documents']]))
print('Human legal accuracy measured:', review['human_accuracy_measured'])


## 4. Exact evidence is necessary, not sufficient
One separate case put an earlier High Court ruling in the current decision section. Another missed the final decision section. Both can still contain copied source text. **Two of three separate cases had serious decision-section failures.** The comparison was performed by the assistant, not independent legal annotators; training overlap is unknown.
The frozen evaluation was not retuned after inspecting it. These now-inspected cases must become development examples if we use them to fix the next version.

In [ ]:
%%project
for item in review['documents']:
    print(item['file'])
    print(item['finding'])
    print()


## 5. A simple ROUGE demonstration
The following invented pair illustrates lexical overlap only. A high score can coexist with a changed condition or outcome.

In [ ]:
%%project
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
reference = 'The appeal is allowed and payment is due within six weeks.'
prediction = 'The appeal is dismissed and payment is due within six weeks.'
display(pd.DataFrame({name: value._asdict() for name, value in scorer.score(reference, prediction).items()}).T)
print('One changed word reverses the outcome. ROUGE alone cannot validate this summary.')


### What to say, and what not to claim
“The recorded role-classification dev macro F1 improved from about 0.39 for logistic regression to 0.65 for the document BiLSTM. The earlier 100-case summarization test reached roughly 0.52 ROUGE-1 F1. The current app has source tracing, but separate review exposed decision errors, so I present it as a research assistant.”

Do not say “83% summary accuracy,” “error-free,” “best model,” or “production ready.” 83% is approximately a role-classification development accuracy, a different task.